# Approach 2 · A5 raw compositional ranking

**Goal.** Reach 50% unseen Top-1 across all 333 candidates, with 60% as a stretch target and an absolute seen/unseen gap of at most 10 percentage points. Report seen Top-1 and harmonic mean alongside unseen Top-1. These are evaluation targets for this experiment.

A4 is the architectural reference. A5 retains DINOv2 ViT-B/14, last-block fine-tuning, Decorrelation, Cross-crop SupCon and Mild Augmentation. It adds disease patch attention, cosine concept heads and weighted ranking with cross-crop recombination. To isolate the model changes in an A4 versus A5 comparison, use the same full-training refit procedure for both. Inference sums the two learned cosine logits for every candidate using the same formula.

## 1. Setup

Place this notebook in the team repository and open it in VS Code with the project Python kernel. Set `DEBUG=True` to check training quickly. Set `DEBUG=False` and Run All to run development, a fresh full-training refit and official evaluation. The run location and final model path are printed by the notebook.

In [ ]:
DEBUG = True

from datetime import datetime, timezone
from pathlib import Path
import copy
import hashlib
import json
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
LOGIT_SCALE = 16.0
HARD_NEGATIVES, PAIR_MARGIN, RANK_WEIGHT, TRANSFER_WEIGHT = 3, 1.0, 0.35, 0.25
RUN_NAME = "approach2_raw_a5_full_refit" + ("_debug" if DEBUG else "")
RECIPE = "A5_Patch_Weighted_Ranking"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = RESULTS_DIR / "development"
FINAL_DIR = RESULTS_DIR / "final"
OFFICIAL_DIR = RESULTS_DIR / "official" / "pair_identity_order"
CHECKPOINT = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID / "model.pt"
LOCK = FINAL_DIR / "selection_lock.json"

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError("Use the shared 323/333 evaluation API.")
print(f"PyTorch {torch.__version__} · {DEVICE} · debug={DEBUG} · run {RUN_ID}")

## 2. Data

Use the same seeded within-pair train/validation split and train-derived proxy pairs as A4. Hold out ten complete training combinations from development training and its ranking objective. Both constituent concepts remain in the fit split. Use five proxy pairs to select the epoch count and five other pairs for a separate development check. Full refit then uses all training CSV rows, including the seen validation images and the train-derived proxy pairs. Official test metadata enters only after the final lock.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def proxy_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Proxy validation needs seen and held-out images.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
train_pairs_device = train_pairs.to(DEVICE)
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "schedule": "CosineAnnealingLR",
    "label_smoothing": SMOOTHING, "trainable_backbone": ["blocks.11", "norm"],
    "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "scoring": "raw_additive_cosine", "logit_scale": LOGIT_SCALE,
    "ranking_weight": RANK_WEIGHT, "ranking_margin": PAIR_MARGIN, "hard_negatives": HARD_NEGATIVES,
    "transfer_ranking_weight": TRANSFER_WEIGHT,
    "pair_weighting": "inverse_sqrt_fit_pair_frequency",
    "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}

## 3. Model and loss

The crop branch reads the [CLS] token. The disease branch learns an attention-weighted average over the 256 spatial tokens at 224 × 224. Both cosine heads use a fixed shared scale of 16. Every candidate receives the same additive score `crop logit + disease logit`.

For each training image, the ranking objective compares the target pair against its three highest scoring **wrong fit pairs** using `softplus(1 + negative score - true score)`. The disease cross-entropy and ranking objective give less frequent fit pairs more weight, using the inverse square root of the training frequency. A cross-crop recombination loss pairs the anchor crop logits with disease logits from same-disease images of other crops in the batch. It ranks the anchor’s true fit pair using that cross-host evidence. Cross-crop SupCon acts on the disease embedding. The optimizer trains the heads and DINOv2's last block and final LayerNorm.

In [ ]:
@torch.no_grad()
def frozen_prefix(backbone, images):
    tokens = backbone.patch_embed(images)
    tokens = backbone._pos_embed(tokens)
    tokens = backbone.norm_pre(backbone.patch_drop(tokens))
    for block in backbone.blocks[:-1]:
        tokens = block(tokens)
    return tokens


def augment_blur(images, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    augmented = images.clone()
    augmented[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                           sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return augmented


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


class CosineHead(nn.Module):
    def __init__(self, width, classes):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, width))
        nn.init.normal_(self.weight, std=0.02)

    def forward(self, features):
        return LOGIT_SCALE * F.linear(F.normalize(features, dim=1), F.normalize(self.weight, dim=1))


class ConceptHeads(nn.Module):
    def __init__(self, width, prefix_tokens):
        super().__init__()
        self.prefix_tokens = prefix_tokens
        self.attention = nn.Sequential(nn.Linear(width, 128), nn.Tanh(), nn.Linear(128, 1, bias=False))
        nn.init.zeros_(self.attention[-1].weight)
        self.disease_projection = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, 256))
        self.crop_head = CosineHead(width, NUM_CROPS)
        self.disease_head = CosineHead(256, NUM_DISEASES)

    def forward(self, tokens):
        crop_feature = tokens[:, 0]
        patches = tokens[:, self.prefix_tokens:]
        attention = self.attention(patches).softmax(dim=1)
        disease_feature = self.disease_projection((patches * attention).sum(dim=1))
        return self.crop_head(crop_feature), self.disease_head(disease_feature), disease_feature


class PatchPairNet(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.heads = ConceptHeads(self.backbone.num_features, self.backbone.num_prefix_tokens)

    def forward(self, images):
        return self.heads(self.backbone.forward_features(images))


class FineTuneTail(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.block = copy.deepcopy(model.backbone.blocks[-1]).requires_grad_(True)
        self.norm = copy.deepcopy(model.backbone.norm).requires_grad_(True)
        self.heads = copy.deepcopy(model.heads)

    def forward(self, tokens):
        return self.heads(self.norm(self.block(tokens)))


@torch.no_grad()
def verify_prefix(model, tail, images):
    for split, regular in zip(tail(frozen_prefix(model.backbone, images)), model(images)):
        torch.testing.assert_close(split, regular, rtol=1e-5, atol=1e-6)


def concept_pair_scores(crop_logits, disease_logits, pairs):
    return crop_logits[:, pairs[:, 0]] + disease_logits[:, pairs[:, 1]]


class PairRankingObjective(nn.Module):
    def __init__(self, fit_pair_rows, pairs):
        super().__init__()
        count = torch.bincount(fit_pair_rows, minlength=len(pairs))
        active = count.gt(0)
        if active.sum().item() < 2:
            raise ValueError("Pair ranking needs at least two fit pairs.")
        self.register_buffer("active", active)
        self.register_buffer("pairs", pairs)
        self.register_buffer("weights", count.clamp_min(1).float().pow(-0.5))
        self.hard_count = min(HARD_NEGATIVES, int(active.sum()) - 1)

    def margins(self, crop, disease, pair_rows):
        scores = concept_pair_scores(crop, disease, self.pairs)
        target = scores.gather(1, pair_rows[:, None])
        negatives = scores.masked_fill(~self.active[None, :], -torch.inf)
        negatives = negatives.scatter(1, pair_rows[:, None], -torch.inf)
        hardest = negatives.topk(self.hard_count, dim=1).values
        return F.softplus(PAIR_MARGIN + hardest - target).mean(dim=1)

    def forward(self, crop, disease, crop_labels, disease_labels, pair_rows):
        weights = self.weights[pair_rows]
        weights = weights / weights.sum()
        crop_ce = F.cross_entropy(crop, crop_labels, label_smoothing=SMOOTHING)
        disease_ce = F.cross_entropy(disease, disease_labels, reduction="none", label_smoothing=SMOOTHING)
        real_rank = torch.dot(weights, self.margins(crop, disease, pair_rows))

        # Attach disease evidence from same-disease images of other crops to the anchor crop.
        # The supervised target remains the anchor's real fit pair.
        donors = disease_labels[:, None].eq(disease_labels) & crop_labels[:, None].ne(crop_labels)
        donor_counts = donors.sum(dim=1)
        recombined = (donors.to(disease.dtype) @ disease) / donor_counts.clamp_min(1)[:, None]
        available = donor_counts > 0
        transfer_weights = self.weights[pair_rows] * available
        transfer_weights = transfer_weights / transfer_weights.sum().clamp_min(1e-8)
        transfer_rank = torch.dot(transfer_weights, self.margins(crop, recombined, pair_rows))

        total = crop_ce + torch.dot(weights, disease_ce) + RANK_WEIGHT * real_rank + TRANSFER_WEIGHT * transfer_rank
        return total, real_rank, transfer_rank, available.sum()

## 4. Development and epoch selection

Select the training length by the raw seen/proxy-unseen Top-1 harmonic mean, then proxy-unseen Top-1, seen Top-1, and earlier epoch. Record the independent proxy check after selecting the epoch. Development starts from the original pretrained DINOv2 weights.

In [ ]:
def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def make_optimizer(tail):
    return torch.optim.AdamW([
        {"params": tail.heads.parameters(), "lr": HEAD_LR},
        {"params": [*tail.block.parameters(), *tail.norm.parameters()], "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


def train_tail(tail, optimizer, objective, tokens, crop, disease, pair_rows):
    crop_logits, disease_logits, disease_features = tail(tokens)
    pair_loss, ranking, transfer, transfers = objective(crop_logits, disease_logits, crop, disease, pair_rows)
    decor = decorrelation(center_logits(crop_logits), center_logits(disease_logits))
    supcon, anchors = cross_crop_supcon(disease_features, crop, disease)
    loss = pair_loss + DECOR_WEIGHT * decor + SUPCON_WEIGHT * supcon
    if not torch.isfinite(loss):
        raise FloatingPointError("Non-finite A5 training loss.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item(), ranking.item(), transfer.item(), anchors.item(), transfers.item()


@torch.no_grad()
def cached_group(backbone, frame):
    rng = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=rng)
    return [(frozen_prefix(backbone, images.to(DEVICE)).cpu(), target.cpu())
            for images, _, _, target in loader]


@torch.no_grad()
def score_group(tail, batches):
    scores, labels = [], []
    for tokens, targets in batches:
        crop, disease, _ = tail(tokens.to(DEVICE))
        scores.append(concept_pair_scores(crop, disease, train_pairs_device).cpu())
        labels.append(targets)
    return torch.cat(scores), torch.cat(labels)


def proxy_metrics(tail, seen_cache, proxy_cache, fit_mask):
    seen_s, seen_y = score_group(tail, seen_cache)
    proxy_s, proxy_y = score_group(tail, proxy_cache)
    seen = top_k_accuracy(seen_s, seen_y, 1)
    proxy = top_k_accuracy(proxy_s, proxy_y, 1)
    held_scores = proxy_s.masked_fill(fit_mask[None, :], -torch.inf)
    winner, held_winner = proxy_s.argmax(dim=1), held_scores.argmax(dim=1)
    blocked = held_winner.eq(proxy_y) & fit_mask[winner]
    return {"seen_top1": seen, "proxy_top1": proxy, "harmonic_top1": harmonic_mean(seen, proxy),
            "proxy_blocked_by_fit_pairs_pct": round(100 * blocked.float().mean().item(), 2)}


def read_final_lock():
    if not LOCK.is_file() or not CHECKPOINT.is_file():
        raise FileNotFoundError("Complete A5 training first.")
    saved = json.loads(LOCK.read_text(encoding="utf-8"))
    if saved["settings"] != settings or saved["checkpoint_sha256"] != sha256_file(CHECKPOINT):
        raise RuntimeError("A5 checkpoint or settings changed after selection.")
    if saved["seen_pairs"] != train_pairs.tolist():
        raise RuntimeError("Training candidate identities changed after selection.")
    return saved


def objective_for(frame):
    targets = torch.as_tensor(frame.pair_row.to_numpy(copy=True), dtype=torch.long)
    return PairRankingObjective(targets, train_pairs).to(DEVICE)


def make_training_model():
    model = PatchPairNet(pretrained=True).to(DEVICE).eval()
    if len(model.backbone.blocks) != 12:
        raise ValueError("Expected the 12-block DINOv2 ViT-B/14.")
    tail = FineTuneTail(model).to(DEVICE).eval()
    return model, tail


def train_epoch(model, tail, optimizer, objective, loader, blur_rng, stage, epoch, epochs, fit_mask=None):
    total_loss, total_ranking, total_transfer = 0.0, 0.0, 0.0
    anchors, transfers, examples = 0, 0, 0
    for step, (images, crops, diseases, pair_rows) in enumerate(loader, 1):
        images = augment_blur(images.to(DEVICE), blur_rng)
        crops, diseases, pair_rows = (tensor.to(DEVICE) for tensor in (crops, diseases, pair_rows))
        if fit_mask is not None:
            keep = fit_mask[pair_rows]
            if not keep.any():
                continue
            images, crops, diseases, pair_rows = images[keep], crops[keep], diseases[keep], pair_rows[keep]
        if epoch == 1 and examples == 0:
            verify_prefix(model, tail, images[:1])
        tokens = frozen_prefix(model.backbone, images)
        loss, ranking, transfer, count, transfer_count = train_tail(
            tail, optimizer, objective, tokens, crops, diseases, pair_rows)
        total_loss += loss * len(images)
        total_ranking += ranking * len(images)
        total_transfer += transfer * transfer_count
        anchors += count
        transfers += transfer_count
        examples += len(images)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"A5 {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)

    return {"epoch": epoch, "images_trained": examples, "train_loss": total_loss / examples,
            "train_ranking_loss": total_ranking / examples,
            "train_transfer_ranking_loss": total_transfer / max(transfers, 1),
            "transfer_anchor_fraction": transfers / examples, "supcon_anchor_fraction": anchors / examples}


def run_development():
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    team.set_seed()
    train, val, test_seen, test_unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if test_seen is not None or test_unseen is not None:
        raise RuntimeError("Development must use only the training CSV.")
    train.pin_memory = False
    model, tail = make_training_model()
    optimizer = make_optimizer(tail)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    fit_frame, fit_mask, fit_device = train.dataset.frame, None, None
    if not DEBUG:
        fit_mask, frames = proxy_split(fit_frame, val.dataset.frame)
        fit_device = fit_mask.to(DEVICE)
        fit_rows = fit_mask.nonzero(as_tuple=True)[0].tolist()
        fit_frame = fit_frame.loc[fit_frame.pair_row.isin(fit_rows)]
        tune = [cached_group(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")]
    objective = objective_for(fit_frame)
    if fit_mask is not None and not torch.equal(objective.active, fit_device):
        raise RuntimeError("Proxy holdout and fit pairs do not match.")

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    best, history = None, []
    for epoch in range(1, EPOCHS + 1):
        row = train_epoch(model, tail, optimizer, objective, train, blur_rng,
                          "development", epoch, EPOCHS, fit_mask=fit_device)
        schedule.step()
        if not DEBUG:
            row.update(proxy_metrics(tail, *tune, fit_mask))
            if best is None or selection_key(row) > selection_key(best["metrics"]):
                best = {"metrics": row.copy(), "weights": snapshot_tail(tail)}
        history.append(row)
        print(row, flush=True)

    pd.DataFrame(history).to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
    if DEBUG:
        return None
    tail.load_state_dict(best["weights"])
    check = [cached_group(model.backbone, frames[key]) for key in ("check_seen", "check_proxy")]
    check_metrics = proxy_metrics(tail, *check, fit_mask)
    pd.DataFrame([check_metrics]).to_csv(DEVELOPMENT_DIR / "independent_proxy_check.csv", index=False)
    selection = {"selection": best["metrics"], "independent_proxy_check": check_metrics,
                 "selection_tune_pair_rows": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
                 "independent_check_pair_rows": sorted(frames["check_proxy"].pair_row.unique().tolist())}
    atomic_json(DEVELOPMENT_DIR / "epoch_selection.json", selection)
    return selection


development = run_development()
if development is not None:
    display(pd.DataFrame([development["selection"], development["independent_proxy_check"]],
                         index=["proxy tune", "independent proxy check"]))

## 5. Full-training refit

Reinitialize from the original pretrained DINOv2 weights and fresh heads. Train on every image in the training CSV for exactly the selected epoch count. Use the development cosine schedule up to that epoch and recompute pair-frequency weights from all training rows. The refit runs for its fixed length and saves the last weights as the official checkpoint. `final/refit_history.csv` records the count of images trained on each epoch.

In [ ]:
def refit_full_train(development):
    if LOCK.is_file():
        raise FileExistsError("This A5 run is complete. Rerun Setup to start again.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development. Start a fresh run.")

    selected_epochs = int(development["selection"]["epoch"])
    if not 1 <= selected_epochs <= EPOCHS:
        raise ValueError("Selected refit length is outside the development epoch budget.")
    team.set_seed()
    model, tail = make_training_model()
    optimizer = make_optimizer(tail)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=team.NUM_WORKERS, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    objective = objective_for(train_full)
    if not objective.active.all():
        raise RuntimeError("Full training data must cover all training candidate pairs.")

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    history = []
    for epoch in range(1, selected_epochs + 1):
        row = train_epoch(model, tail, optimizer, objective, loader, blur_rng,
                          "full refit", epoch, selected_epochs)
        if row["images_trained"] != len(train_full):
            raise RuntimeError("A refit epoch did not process every training image.")
        schedule.step()
        history.append(row)
        print(row, flush=True)

    model.backbone.blocks[-1].load_state_dict(tail.block.state_dict())
    model.backbone.norm.load_state_dict(tail.norm.state_dict())
    model.heads.load_state_dict(tail.heads.state_dict())
    model.cpu()
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "training_pairs": len(train_pairs), "epochs": selected_epochs, "cosine_t_max": EPOCHS}
    checkpoint_metadata = {"settings": settings, "recipe": RECIPE, "selection": development["selection"],
                           "seen_pairs": train_pairs.tolist(), "refit": refit}
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(FINAL_DIR / "refit_history.csv", index=False)
    atomic_save(CHECKPOINT, {"state_dict": model.state_dict(), **checkpoint_metadata})
    locked = {**development, **checkpoint_metadata, "checkpoint_sha256": sha256_file(CHECKPOINT),
              "checkpoint_bytes": CHECKPOINT.stat().st_size}
    atomic_json(LOCK, locked)
    return locked


run = None if DEBUG else refit_full_train(development)
if run is not None:
    print(f"Full refit locked: {run['refit']['images_per_epoch']} images per epoch, "
          f"{run['refit']['epochs']} epochs.")

## 6. Official evaluation

Use the locked weights and the same 333 candidates for every official image. Build the final shortlist from the 323 training pairs and 10 unseen concept identities. Finish all raw predictions before joining per-image ground truth. The 10-candidate score is a diagnostic. It measures discrimination among unseen pairs when seen pairs are removed from the candidate pool.

Read `official_comparison.csv` for the three joint evaluations and the target status. `unseen_ranking_diagnostics.csv` and `unseen_per_pair.csv` show whether unseen mistakes come from crop/disease errors or seen pairs outranking the correct unseen pair. The full model is saved to `checkpoints/approach2_raw_a5_full_refit/<run_id>/model.pt`. Development, final selection and official reports are under `results/approach2_raw_a5_full_refit/<run_id>/`.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    generator = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=generator)
    mean, std = (torch.tensor(values).view(1, 3, 1, 1) for values in team.NORM_STATS["imagenet"])
    pair_indices = pairs.to(DEVICE)
    scores, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop, disease, _ = model(images.to(DEVICE))
        batch = concept_pair_scores(crop, disease, pair_indices).cpu()
        if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
            raise ValueError("Invalid raw pair scores.")
        scores.append(batch)
        concepts.append(torch.stack((crop.argmax(dim=1), disease.argmax(dim=1)), dim=1).cpu())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(scores), torch.cat(concepts)


def official_evaluation():
    if DEBUG or run is None or read_final_lock() != run:
        raise RuntimeError("Run A5 development and lock the checkpoint before official evaluation.")
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    report_path = OFFICIAL_DIR / "official_report.json"
    if report_path.is_file():
        existing = json.loads(report_path.read_text(encoding="utf-8"))
        if existing["lock"] != run:
            raise RuntimeError("An existing report belongs to a different experiment.")
        return existing

    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    model = PatchPairNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model.to(DEVICE).eval()

    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    seen_files, unseen_files = map(official_filenames, paths)
    official_files = set(seen_files) | set(unseen_files)
    if set(seen_files) & set(unseen_files) or official_files & set(train_full.image):
        raise ValueError("Training and official split filenames must be disjoint.")

    seen_s, _ = predict_official(model, pairs, seen_files, "seen")
    unseen_s, unseen_heads = predict_official(model, pairs, unseen_files, "unseen")
    boundary = len(train_pairs)
    seen_pred, unseen_pred = top_predictions(seen_s), top_predictions(unseen_s)
    seen_only = top_predictions(seen_s[:, :boundary])
    unseen_only = top_predictions(unseen_s[:, boundary:], offset=boundary)
    robust_pred = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            scores, _ = predict_official(model, pairs, seen_files, condition, transform)
            robust_pred[condition] = scores.topk(1, dim=1).indices

    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], seen_files, rows, seen=True)
    y_unseen = official_targets(paths[1], unseen_files, rows, seen=False)
    y_all = torch.cat((y_seen, y_unseen))
    raw = {"recipe": RECIPE, "scoring": "raw", "epoch": run["selection"]["epoch"], "candidates": len(pairs)}
    for k in (1, 5):
        raw[f"seen_top{k}"] = prediction_accuracy(seen_pred[k], y_seen)
        raw[f"unseen_top{k}"] = prediction_accuracy(unseen_pred[k], y_unseen)
        raw[f"overall_top{k}"] = prediction_accuracy(torch.cat((seen_pred[k], unseen_pred[k])), y_all)
    raw["harmonic_top1"] = harmonic_mean(raw["seen_top1"], raw["unseen_top1"])
    raw["seen_minus_unseen_top1_pp"] = round(raw["seen_top1"] - raw["unseen_top1"], 2)
    raw["abs_seen_unseen_gap_pp"] = abs(raw["seen_minus_unseen_top1_pp"])
    raw["meets_unseen_50pct"] = raw["unseen_top1"] >= 50
    raw["meets_unseen_60pct"] = raw["unseen_top1"] >= 60
    raw["meets_50pct_and_gap_at_most_10pp"] = raw["meets_unseen_50pct"] and raw["abs_seen_unseen_gap_pp"] <= 10

    restricted = []
    for name, prediction, labels, size in (
        ("seen_only", seen_only, y_seen, boundary),
        ("unseen_only", unseen_only, y_unseen, len(pairs) - boundary),
    ):
        restricted.append({"recipe": RECIPE, "scoring": "raw", "split": name, "candidates": size,
                           **{f"top{k}": prediction_accuracy(prediction[k], labels) for k in (1, 5)}})
    robustness = []
    for condition, prediction in robust_pred.items():
        accuracy = prediction_accuracy(prediction, y_seen)
        robustness.append({"condition": condition, "seen_top1": accuracy,
                           "drop_pp": round(raw["seen_top1"] - accuracy, 2)})

    joint_correct = unseen_pred[1].squeeze(1).eq(y_unseen)
    only_correct = unseen_only[1].squeeze(1).eq(y_unseen)
    predicted_seen = unseen_pred[1].squeeze(1).lt(boundary)
    blocked = only_correct & predicted_seen
    crop_hit, disease_hit = unseen_heads.eq(pairs[y_unseen]).T
    true_score = unseen_s.gather(1, y_unseen[:, None]).squeeze(1)
    margins = true_score - unseen_s[:, :boundary].amax(dim=1)
    ranking = {"unseen_images": len(y_unseen), "joint_correct": int(joint_correct.sum()),
               "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicted_seen.sum()),
               "correct_within_unseen_but_blocked_by_seen": int(blocked.sum()),
               "crop_head_top1": round(100 * crop_hit.float().mean().item(), 2),
               "disease_head_top1": round(100 * disease_hit.float().mean().item(), 2),
               "median_true_pair_minus_top_seen_score": float(margins.median())}
    per_pair = []
    for index in range(boundary, len(pairs)):
        mask = y_unseen.eq(index)
        support = int(mask.sum())
        crop_id, disease_id = pairs[index].tolist()
        per_pair.append({"crop": crop_id, "disease": disease_id, "support": support,
                         "joint_correct": int((mask & joint_correct).sum()),
                         "unseen_only_correct": int((mask & only_correct).sum()),
                         "blocked_by_seen": int((mask & blocked).sum())})

    # Shared confusion display uses the team candidate ordering after scoring.
    canonical = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if canonical.keys() != rows.keys():
        raise ValueError("Candidate identities differ from the shared vocabulary.")
    to_canonical = torch.tensor([canonical[tuple(pair)] for pair in pairs.tolist()])
    predictions = F.one_hot(to_canonical[seen_pred[1].squeeze(1)], num_classes=len(pairs)).float()
    stem = (OFFICIAL_DIR / f"{RECIPE}_raw").relative_to(team.RESULTS_DIR).as_posix()
    save_confusion(predictions, to_canonical[y_seen], stem, final_evaluation=True)

    for name, records in (("official_comparison", [raw]), ("restricted_candidates", restricted),
                          ("official_robustness", robustness), ("unseen_ranking_diagnostics", [ranking]),
                          ("unseen_per_pair", per_pair)):
        pd.DataFrame(records).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    report = {"lock": run, "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist(),
              "test_counts": {"seen": len(y_seen), "unseen": len(y_unseen)}, "comparison": [raw],
              "restricted_candidates": restricted, "robustness": robustness,
              "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair}
    atomic_json(report_path, report)
    return report


if DEBUG:
    print(f"Debug development logs: {DEVELOPMENT_DIR.resolve()}")
else:
    report = official_evaluation()
    display(pd.DataFrame(report["comparison"]))
    display(pd.DataFrame([report["unseen_ranking_diagnostics"]]))
    print(f"Full A5 checkpoint: {CHECKPOINT.resolve()}")
    print(f"Official reports: {OFFICIAL_DIR.resolve()}")